# Credit scoring explanations

## Setup API requests

In [ ]:
import requests
from openai import OpenAI, DefaultHttpxClient

base_url = "https://inference.airi.net:46783/v1"
client = OpenAI(
    base_url=base_url,
    api_key=api_key,
    http_client=DefaultHttpxClient(verify=False)
)
models = client.models.list()
models

SyncPage[Model](data=[Model(id='Deepseek-ai/DeepSeek-R1-Distill-Llama-70B', created=1755864677, object='model', owned_by='vllm', root='Deepseek-ai/DeepSeek-R1-Distill-Llama-70B', parent=None, max_model_len=32768, permission=[{'id': 'modelperm-cb72ca62fdab4fd288b1f618583767ff', 'object': 'model_permission', 'created': 1755864677, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]), Model(id='Qwen/Qwen2.5-72B-Instruct', created=1755864677, object='model', owned_by='vllm', root='Qwen/Qwen2.5-72B-Instruct', parent=None, max_model_len=16384, permission=[{'id': 'modelperm-aed556a834724292a33afb40e13350dd', 'object': 'model_permission', 'created': 1755864677, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '

In [33]:
def generate_response(messages, model: str = "Deepseek-ai/DeepSeek-R1-Distill-Llama-70B"):
    response = client.chat.completions.create(
        model=model,
        # logprobs=True,
        # top_logprobs=10,
        messages=messages,
        # max_tokens=8192
    )
    return response

## Formate prompts

### Read dataset and add some features

In [34]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis

df = pd.read_csv('../data/all.csv')

# Replace NaNs with new class
df['term_id'] = df['term_id'].fillna(-1)  # or any unique sentinel value

# Ensure datetime is parsed
df['tr_datetime'] = pd.to_datetime(df['tr_datetime'], errors='coerce')
assert np.issubdtype(df['tr_datetime'].dtype, np.datetime64), "tr_datetime must be parsed"

# Sort by datetime
df.sort_values(['customer_id', 'tr_datetime'], ascending=[True, True], inplace=True)

df

,customer_id,tr_datetime,mcc_code,tr_type,amount,term_id,gender,tr_type_desc,mcc_code_desc
0,0,2000-01-01 10:23:26,0,0,-2245.92,-1,1,Оплата услуги. Банкоматы СБ РФ,"Звонки с использованием телефонов, считывающих..."
1,0,2000-01-02 10:19:29,1,1,56147.89,-1,1,Взнос наличных через АТМ (в своем тер.банке),Финансовые институты — снятие наличности автом...
2,0,2000-01-02 10:20:56,2,2,-56147.89,-1,1,Списание с карты по операции “перевода с карты...,Денежные переводы
3,0,2000-01-02 10:39:54,3,3,-1392.47,-1,1,Покупка. POS ТУ СБ РФ,"Различные продовольственные магазины — рынки, ..."
4,0,2000-01-03 15:33:42,3,3,-920.83,-1,1,Покупка. POS ТУ СБ РФ,"Различные продовольственные магазины — рынки, ..."
...,...,...,...,...,...,...,...,...,...
3751078,8399,2001-03-29 16:03:02,3,3,-5176.84,10217113,0,Покупка. POS ТУ СБ РФ,"Различные продовольственные магазины — рынки, ..."
3751079,8399,2001-03-30 10:54:59,10,3,-1652.77,022915,0,Покупка. POS ТУ СБ РФ,"Бакалейные магазины, супермаркеты"
3751080,8399,2001-03-30 14:23:59,3,3,-4687.23,10217113,0,Покупка. POS ТУ СБ РФ,"Различные продовольственные магазины — рынки, ..."
3751081,8399,2001-03-30 16:11:53,4,6,-4491.83,RU570124,0,Покупка. POS ТУ Россия,Станции техобслуживания


In [35]:

# ─────────────────────────────────────
# Feature Engineering Per Customer
# ─────────────────────────────────────

# 1. Basic transaction stats + counts of unique categorical features
tx_stats = df.groupby('customer_id').agg({
    'amount': ['count', 'sum', 'mean', 'std', 'min', 'max'],
    'term_id': pd.Series.nunique,
    'mcc_code': pd.Series.nunique,
    'tr_type': pd.Series.nunique
})

# Flatten multiindex columns
tx_stats.columns = [f'base_{k}_{stat}' for k, stat in tx_stats.columns]
tx_stats.reset_index(inplace=True)

# 2. Positive vs negative transaction breakdown
df['is_positive'] = (df['amount'] > 0).astype(int)
df['is_negative'] = (df['amount'] < 0).astype(int)

pos_neg_stats = df.groupby('customer_id').agg({
    'is_positive': 'mean',
    'is_negative': 'mean'
}).rename(columns={
    'is_positive': 'share_positive_txn',
    'is_negative': 'share_negative_txn'
}).reset_index()

# 3. Enhanced amount aggregations
df['amount_positive'] = df['amount'].where(df['amount'] > 0, 0)
df['amount_negative'] = df['amount'].where(df['amount'] < 0, 0)

agg_funcs = {
    'amount': ['median',
               lambda x: np.percentile(x, 25),
               lambda x: np.percentile(x, 75),
               skew,
               kurtosis
               ],
    'amount_positive': ['sum', 'count'],
    'amount_negative': ['sum', 'count']
}

amount_stats = df.groupby('customer_id').agg(agg_funcs)

# Rename columns for clarity
amount_stats.columns = [
    'amount_median',
    'amount_pct25',
    'amount_pct75',
    'amount_skew',
    'amount_kurtosis',
    'amount_positive_sum',
    'amount_positive_count',
    'amount_negative_sum',
    'amount_negative_count'
]

# Calculate ratios and shares

amount_stats['amount_pos_count_share'] = (
    amount_stats['amount_positive_count'] / (amount_stats['amount_positive_count'] + amount_stats['amount_negative_count'] + 1e-9)
)

# Handle inf and NaN
amount_stats.replace([np.inf, -np.inf], 0, inplace=True)
amount_stats.fillna(0, inplace=True)
amount_stats.reset_index(inplace=True)

# 3. Temporal patterns
df['hour'] = df['tr_datetime'].dt.hour
df['minute'] = df['tr_datetime'].dt.minute
df['weekday'] = df['tr_datetime'].dt.weekday
df['day'] = (df['tr_datetime'] - pd.to_datetime("2000-01-01")).dt.days
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)
df['days_since_last_txn'] = df.groupby('customer_id')['day'].transform(lambda x: x.max() - x)

weekday_map = {0: 'Понедельник', 1: 'Вторник', 2: 'Среда', 3: 'Четверг', 4: 'Пятница', 5: 'Суббота', 6: 'Воскресенье'}

df['weekday_name'] = df['weekday'].map(weekday_map)

temporal_stats = df.groupby('customer_id').agg({
    'minute': ['mean', 'std', 'min', 'max'],
    'weekday': ['mean', 'std', pd.Series.nunique],
    'is_weekend': 'mean',
    'day': ['min', 'max', pd.Series.nunique],
    'days_since_last_txn': 'mean'
})

temporal_stats.columns = [f'temp_{k}_{stat}' for k, stat in temporal_stats.columns]
temporal_stats.reset_index(inplace=True)

weekday_freq = (
    df.groupby(['customer_id', 'weekday_name'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['weekday_name'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='weekday_freq_dict')
)

hour_freq = (
    df.groupby(['customer_id', 'hour'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['hour'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='hour_freq_dict')
)

temporal_stats = temporal_stats \
    .merge(weekday_freq, on='customer_id', how='left') \
    .merge(hour_freq, on='customer_id', how='left') 

# 4. MCC code frequency as dict
mcc_freq = (
    df.groupby(['customer_id', 'mcc_code_desc'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['mcc_code_desc'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='mcc_freq_dict')
)

# 5. Transaction type frequency as dict
trtype_freq = (
    df.groupby(['customer_id', 'tr_type_desc'])
    .size()
    .reset_index(name='count')
    .groupby('customer_id')
    .apply(lambda x: dict(sorted(zip(x['tr_type_desc'], x['count']), key=lambda kv: kv[1], reverse=True)))
    .reset_index(name='trtype_freq_dict')
)

# 6. Transaction density
txn_density = df.groupby('customer_id')['day'].agg(['count', 'nunique'])
txn_density['txn_per_day'] = txn_density['count'] / txn_density['nunique']
txn_density = txn_density[['txn_per_day']].reset_index()


# ─────────────────────────────────────
# Merge all features
# ─────────────────────────────────────
features = tx_stats \
    .merge(pos_neg_stats, on='customer_id', how='left') \
    .merge(amount_stats, on='customer_id', how='left') \
    .merge(temporal_stats, on='customer_id', how='left') \
    .merge(txn_density, on='customer_id', how='left') \
    .merge(mcc_freq, on='customer_id', how='left') \
    .merge(trtype_freq, on='customer_id', how='left') 

# Add target
target = df[['customer_id', 'gender']].drop_duplicates()
features = features.merge(target, on='customer_id', how='left')

# Final data
X = features.drop(columns=['customer_id', 'gender'])
y = features['gender']

/var/folders/5z/wq1n27lj75g6vkwl69d2b5_c0000gp/T/ipykernel_12861/1756435468.py:98: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(sorted(zip(x['weekday_name'], x['count']), key=lambda kv: kv[1], reverse=True)))
/var/folders/5z/wq1n27lj75g6vkwl69d2b5_c0000gp/T/ipykernel_12861/1756435468.py:107: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: dict(sorted(zip(x['hour'], x['count']), k

In [36]:
features

,customer_id,base_amount_count,base_amount_sum,base_amount_mean,base_amount_std,base_amount_min,base_amount_max,base_term_id_nunique,base_mcc_code_nunique,base_tr_type_nunique,...,temp_day_min,temp_day_max,temp_day_nunique,temp_days_since_last_txn_mean,weekday_freq_dict,hour_freq_dict,txn_per_day,mcc_freq_dict,trtype_freq_dict,gender
0,0,1046,-14337300.08,-13706.787839,64290.843681,-898366.31,224591.58,111,16,13,...,0,445,317,191.652008,"{'Воскресенье': 185, 'Среда': 168, 'Суббота': ...","{0: 207, 12: 112, 11: 91, 13: 87, 14: 82, 15: ...",3.299685,{'Различные продовольственные магазины — рынки...,"{'Покупка. POS ТУ СБ РФ': 537, 'Покупка. POS Т...",1
1,1,792,-59123105.46,-74650.385682,327272.840595,-2245915.77,2695098.93,190,39,15,...,0,451,230,248.593434,"{'Среда': 137, 'Вторник': 130, 'Суббота': 128,...","{16: 87, 14: 74, 11: 67, 17: 67, 15: 62, 12: 6...",3.443478,{'Финансовые институты — снятие наличности авт...,"{'Покупка. POS ТУ СБ РФ': 267, 'Выдача наличны...",1
2,2,132,-6480814.62,-49097.080455,94387.619200,-561478.94,112295.79,37,4,4,...,37,456,113,199.939394,"{'Среда': 24, 'Суббота': 23, 'Понедельник': 19...","{2: 24, 11: 14, 12: 13, 6: 12, 10: 12, 8: 10, ...",1.168142,{'Финансовые институты — снятие наличности авт...,"{'Выдача наличных в АТМ Сбербанк России': 74, ...",1
3,3,855,-20145664.14,-23562.180281,150274.377040,-2582803.14,1289155.65,229,50,17,...,0,456,335,225.304094,"{'Суббота': 174, 'Воскресенье': 151, 'Вторник'...","{12: 119, 11: 113, 13: 104, 10: 74, 9: 72, 7: ...",2.552239,{'Различные продовольственные магазины — рынки...,"{'Покупка. POS ТУ СБ РФ': 480, 'Покупка. POS Т...",1
4,4,230,-3600272.66,-15653.359391,110800.014304,-1122957.89,1122957.89,21,12,8,...,0,456,171,218.726087,"{'Суббота': 44, 'Пятница': 40, 'Понедельник': ...","{9: 41, 5: 30, 11: 30, 8: 27, 6: 26, 10: 23, 7...",1.345029,{'Финансовые институты — снятие наличности авт...,"{'Выдача наличных в АТМ Сбербанк России': 151,...",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8395,8395,337,-628312.02,-1864.427359,197631.793313,-1122957.89,2358211.56,66,25,13,...,5,456,182,249.029674,"{'Среда': 62, 'Воскресенье': 52, 'Понедельник'...","{0: 113, 18: 27, 20: 22, 21: 19, 17: 18, 19: 1...",1.851648,"{'Бакалейные магазины, супермаркеты': 123, 'Фи...","{'Покупка. POS ТУ Россия': 86, 'Покупка. POS Т...",1
8396,8396,44,-2713492.98,-61670.295000,98166.113010,-260526.23,168443.68,14,3,4,...,2,437,30,201.818182,"{'Воскресенье': 14, 'Вторник': 8, 'Суббота': 7...","{11: 7, 10: 5, 12: 5, 6: 4, 9: 4, 14: 3, 5: 2,...",1.466667,{'Финансовые институты — снятие наличности авт...,"{'Выдача наличных в АТМ Сбербанк России': 20, ...",0
8397,8397,289,19023.27,65.824464,390381.328208,-1122957.89,2695098.93,35,9,12,...,13,456,130,250.117647,"{'Пятница': 63, 'Воскресенье': 58, 'Четверг': ...","{17: 29, 7: 26, 15: 24, 9: 22, 11: 22, 13: 19,...",2.223077,{'Финансовые институты — снятие наличности авт...,"{'Выдача наличных в АТМ Сбербанк России': 73, ...",1
8398,8398,141,-1587648.01,-11259.914965,374252.254848,-673774.73,3575497.91,71,16,14,...,32,451,79,171.425532,"{'Пятница': 34, 'Суббота': 27, 'Вторник': 23, ...","{0: 28, 9: 19, 15: 14, 10: 11, 14: 10, 17: 10,...",1.784810,{'Финансовые институты — снятие наличности авт...,"{'Покупка. POS ТУ СБ РФ': 34, 'Выдача наличных...",0


### Add readable stats per user

In [37]:
readable_features = features.drop(columns=["gender", "customer_id"]).rename(columns={
    "base_amount_count": "Количество всех транзакций клиента",
    "base_amount_sum": "Общая сумма всех транзакций клиента",
    "base_amount_mean": "Средняя сумма одной транзакции",
    "base_amount_std": "Стандартное отклонение суммы транзакций",
    "base_amount_min": "Минимальная сумма транзакции",
    "base_amount_max": "Максимальная сумма транзакции",
    "base_term_id_nunique": "Количество уникальных терминалов, где проводились транзакции",
    "base_mcc_code_nunique": "Количество уникальных MCC кодов (категорий трат)",
    "base_tr_type_nunique": "Количество уникальных типов транзакций",
    "share_positive_txn": "Доля транзакций с положительной суммой (поступления)",
    "share_negative_txn": "Доля транзакций с отрицательной суммой (расходы)",
    "amount_median": "Медианная сумма транзакции",
    "amount_pct25": "25-й перцентиль суммы транзакции",
    "amount_pct75": "75-й перцентиль суммы транзакции",
    "amount_skew": "Асимметрия распределения сумм транзакций",
    "amount_kurtosis": "Эксцесс (острота) распределения сумм транзакций",
    "amount_positive_sum": "Общая сумма всех положительных транзакций",
    "amount_positive_count": "Количество положительных транзакций",
    "amount_negative_sum": "Общая сумма всех отрицательных транзакций",
    "amount_negative_count": "Количество отрицательных транзакций",
    "amount_pos_count_share": "Доля положительных транзакций по количеству",
    "temp_minute_mean": "Среднее значение минут проведения транзакций (например, по времени суток)",
    "temp_minute_std": "Стандартное отклонение минут проведения транзакций",
    "temp_minute_min": "Минимальное значение минут проведения транзакций",
    "temp_minute_max": "Максимальное значение минут проведения транзакций",
    "temp_weekday_mean": "Средний день недели транзакций (например, ближе к будням или выходным)",
    "temp_weekday_std": "Стандартное отклонение по дням недели транзакций",
    "temp_weekday_nunique": "Количество уникальных дней недели, в которые были транзакции",
    "temp_is_weekend_mean": "Средняя доля транзакций, которые происходят в выходные",
    "temp_day_min": "Минимальный календарный день транзакций",
    "temp_day_max": "Максимальный календарный день транзакций",
    "temp_day_nunique": "Количество уникальных календарных дней с транзакциями",
    "temp_days_since_last_txn_mean": "Среднее количество дней между транзакциями",
    "txn_per_day": "Среднее количество транзакций в день",
    "mcc_freq_dict": "Словарь частот по категориям MCC кодов",
    "trtype_freq_dict": "Словарь частот по типам транзакций",
})

# stats_descriptions_list = readable_features.apply(lambda x: ", ".join([f"{k} = {v}" for k, v in x.items()]), axis=1).to_list()
stats_descriptions_list = readable_features.to_dict(orient='records')

print(stats_descriptions_list[0])

{'Количество всех транзакций клиента': 1046, 'Общая сумма всех транзакций клиента': -14337300.08, 'Средняя сумма одной транзакции': -13706.787839388146, 'Стандартное отклонение суммы транзакций': 64290.84368059456, 'Минимальная сумма транзакции': -898366.31, 'Максимальная сумма транзакции': 224591.58, 'Количество уникальных терминалов, где проводились транзакции': 111, 'Количество уникальных MCC кодов (категорий трат)': 16, 'Количество уникальных типов транзакций': 13, 'Доля транзакций с положительной суммой (поступления)': 0.03919694072657744, 'Доля транзакций с отрицательной суммой (расходы)': 0.9608030592734226, 'Медианная сумма транзакции': -3368.87, '25-й перцентиль суммы транзакции': -9856.7625, '75-й перцентиль суммы транзакции': -1684.44, 'Асимметрия распределения сумм транзакций': -6.131037631143713, 'Эксцесс (острота) распределения сумм транзакций': 62.336715164974834, 'Общая сумма всех положительных транзакций': 3599529.2399999998, 'Количество положительных транзакций': 1046

### Add readable table

In [38]:
from tqdm import tqdm
tqdm.pandas()

column2readable_name = {
    "tr_datetime": "дата и время проведения транзакции",
    "amount": "сумма транзакции",
    "tr_type_desc": "описание типа транзакции (например, покупка, возврат и т.д.)",
    "mcc_code_desc": "описание MCC-кода, определяющего категорию торговой точки",
    "hour": "час проведения транзакции",
    "minute": "минута проведения транзакции",
    "weekday": "день недели, когда произошла транзакция",
    "day": "день месяца транзакции",
    "is_weekend": "флаг, указывающий, была ли транзакция в выходной день",
    "days_since_last_txn": "количество дней, прошедших с последней транзакции этого клиента"
}

df_readable_columns = df.drop(columns=["term_id", "gender", "mcc_code", "tr_type", "amount_positive", "amount_negative"]).rename(columns=column2readable_name)
markdown_tables_last_100_list = df_readable_columns.groupby("customer_id").progress_apply(lambda x: x[-50:].to_markdown()).to_list()
markdown_tables_first_100_list = df_readable_columns.groupby("customer_id").progress_apply(lambda x: x[:50].to_markdown()).to_list()
print(markdown_tables_first_100_list[0])

100%|█████████▉| 8396/8400 [01:44<00:00, 83.43it/s]/Users/savkin/miniconda3/envs/sber_interp/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return getattr(df, df_function)(wrapper, **kwargs)
100%|█████████▉| 8393/8400 [01:44<00:00, 86.87it/s] /Users/savkin/miniconda3/envs/sber_interp/lib/python3.10/site-packages/tqdm/std.py:917: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


|    |   customer_id | дата и время проведения транзакции   |   сумма транзакции | описание типа транзакции (например, покупка, возврат и т.д.)                                     | описание MCC-кода, определяющего категорию торговой точки                                                                                                  |   is_positive |   is_negative |   час проведения транзакции |   минута проведения транзакции |   день недели, когда произошла транзакция |   день месяца транзакции |   флаг, указывающий, была ли транзакция в выходной день |   количество дней, прошедших с последней транзакции этого клиента | weekday_name   |
|---:|--------------:|:-------------------------------------|-------------------:|:-------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------------------------------|--------------

### Construct prompts

In [46]:
def generate_reasoning_prompt(md_table_first: str, md_table_last: str,  aggregates: dict, true_label: str) -> dict:
    """
    Генерирует промпт для reasoning модели с разделением на системный и пользовательский контекст.

    Args:
        md_table (str): Таблица транзакций клиента в формате Markdown.
        aggregates (dict): Агрегированные показатели по всем транзакциям клиента.
        true_label (str): Истинная метка пола ("Мужчина" или "Женщина").

    Returns:
        dict: Словарь с ключами "system" и "user".
    """

    # Форматируем агрегаты в строку
    agg_str = ", ".join([f"{k} = {v}" for k, v in aggregates.items()])

    system_prompt = "Ты эксперт по анализу поведения клиентов по их финансовым транзакциям. Твоя задача — рассуждать и объяснять, почему клиент относится к определенному полу на основе транзакций и агрегированных данных."

    user_prompt = f"""
Данные клиента:

1. Сэмпл транзакций (Markdown таблица):

Первые 50 транзакций:
{md_table_first}

Последние 50 транзакций:
{md_table_last}

2. Агрегированные показатели:
{agg_str}

Задача:
- Проанализировать транзакции и агрегаты.
- Рассуждать, какие признаки поведения (тип транзакций, суммы, MCC-коды, распределение расходов и т.д.) могут указывать на пол.
- Выдать подробное объяснение, почему пол такой-то.
- Объяснение должно быть логичным, опираться на данные, без догадок.

Формат ответа:
1. "Пол клиента: Мужчина/Женщина" - ответ должен быть только из этих двух вариантов, нельзя отвечать незнаю, не уверен и т.д.
2. Подробное объяснение:
- [пункт 1]
- [пункт 2]
- ...
"""

    return {"system_prompt": system_prompt, "user_prompt": user_prompt}

generate_reasoning_prompt(markdown_tables_first_100_list[0], markdown_tables_last_100_list[0], stats_descriptions_list[0], "Мужчина")

{'system_prompt': 'Ты эксперт по анализу поведения клиентов по их финансовым транзакциям. Твоя задача — рассуждать и объяснять, почему клиент относится к определенному полу на основе транзакций и агрегированных данных.',
 'user_prompt': '\nДанные клиента:\n\n1. Сэмпл транзакций (Markdown таблица):\n\nПервые 50 транзакций:\n|    |   customer_id | дата и время проведения транзакции   |   сумма транзакции | описание типа транзакции (например, покупка, возврат и т.д.)                                     | описание MCC-кода, определяющего категорию торговой точки                                                                                                  |   is_positive |   is_negative |   час проведения транзакции |   минута проведения транзакции |   день недели, когда произошла транзакция |   день месяца транзакции |   флаг, указывающий, была ли транзакция в выходной день |   количество дней, прошедших с последней транзакции этого клиента | weekday_name   |\n|---:|--------------:|:---

In [47]:
customers_info = [{
    "transactions_first_100": t1, 
    "transactions_last_100": t2, 
    "aggregates": agg, 
    "gender": gender,
    **generate_reasoning_prompt(t1, t2, agg, gender),
    } 
                  for t1, t2, agg, gender in tqdm(zip(markdown_tables_first_100_list, markdown_tables_last_100_list, stats_descriptions_list, y.to_list()))]

import json
with open("../data/customers_info.json", "w") as f:
    json.dump(customers_info, f, ensure_ascii=False, indent=4)

8400it [00:05, 1585.39it/s]


## Inference models

In [49]:
import json
with open("../data/customers_info.json", "r") as f:
    customers_info = json.load(f)

In [45]:
import re

for customer in customers_info[:5]:
    messages = [
        {"role": "system", "content": customer["system_prompt"]},
        {"role": "user", "content": customer["user_prompt"]}
    ]
    response = generate_response(messages)
    response_text = response.choices[0].message.content
    match = re.search(r"(мужчина|женщина)", response_text, re.IGNORECASE)

    predicted_gender = match.group(1).capitalize() if match else None
    print(response_text)
    print("Итого: ", predicted_gender, customer["gender"])
    print("###############################################")



1. Краткий вывод: "Пол клиента: Женщина"

2. Подробное объяснение:
- **Частые покупки в продовольственных магазинах**: Наибольшая доля транзакций приходится на категорию "Различные продовольственные магазины", что может быть связано с регулярным шопингом для домашнего хозяйства, более характерным для женщин.
- **Операции по снятию наличных и переводам**: Высокая активность в снятии наличных и денежных переводах может указывать на управление домашним бюджетом, что традиционно часто лежит на женщине.
- **Покупки в аптеках и универсальных магазинах**: Эти категории часто связаны с заботой о здоровье и бытовых нуждах, что также более типично для женщин.
- **Распределение по дням недели**: Транзакции чаще совершаются в выходные дни, что может быть связано с планированием шопинга и домашних дел на выходных.
- **Время проведения транзакций**: Большинство операций происходит в утренние и дневные часы, что соответствует стилю жизни, связанному с управлением домашним хозяйством.
Итого:  Женщин